In [10]:
!pip install -q streamlit pyngrok google-genai

In [11]:
from pyngrok import ngrok
import os
from dotenv import load_dotenv

load_dotenv()
ngrok_token = os.getenv('NGROK_TOKEN')

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
    print("ngrok token berhasil dikonfigurasi!")
else:
    print("Peringatan: NGROK_TOKEN tidak ditemukan.")

ngrok token berhasil dikonfigurasi!


In [12]:
import subprocess
import time
import os

def run_streamlit(filename, port=8501):
    # Bersihkan proses lama agar port tidak bentrok
    try:
        res = subprocess.run(f"netstat -ano | findstr :{port}", shell=True, capture_output=True, text=True)
        for line in res.stdout.splitlines():
            if "LISTENING" in line:
                pid = line.strip().split()[-1]
                subprocess.run(f"taskkill /F /PID {pid}", shell=True, capture_output=True)
    except Exception:
        pass

    # Tutup semua tunnel ngrok
    try:
        ngrok.kill()
    except Exception:
        pass

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    try:
        public_url = ngrok.connect(port)
        print(f"Streamlit berjalan di URL Publik: {public_url}")
    except Exception as e:
        print(f"Gagal koneksi ngrok. Streamlit berjalan di http://localhost:{port}")

    return proc

# Streamlit basic


In [24]:
%%writefile streamlit_app_basic.py
import streamlit as st
import pandas as pd
import numpy as np
import time

st.title("Streamlit Basic Tutorial")

st.write("Ini adalah demo komponen-komponen dasar yang tersedia di Streamlit.")
st.write("Setiap section di bawah menunjukkan cara kerja satu komponen.")

st.header("1. Text Input")
user_input = st.text_input("Masukkan namamu", "Ketik di sini...")

st.header("2. Button")
if st.button("Klik aku!"):
    st.write("Tombol diklik!")

st.header("3. Checkbox")
show_content = st.checkbox("Tampilkan pesan rahasia")
if show_content:
    st.write("Pesan rahasia: kamu keren!")

st.header("4. Selectbox")
option = st.selectbox("Pilih warna favoritmu", ("Merah", "Biru", "Hijau", "Kuning"))

st.header("5. Slider")
age = st.slider("Berapa umurmu?", 0, 100, 25)

st.header("6. Progress Bar")
progress_bar = st.progress(0)
for i in range(100):
    time.sleep(0.01)
    progress_bar.progress(i + 1)

st.header("7. Sidebar")
with st.sidebar:
    st.header("Panel Samping")
    if st.button("Tombol di Sidebar"):
        st.write("Sidebar diklik!")

st.header("8. Columns")
col1, col2 = st.columns(2)
with col1:
    st.button("Tombol di kolom kiri")
with col2:
    st.button("Tombol di kolom kanan")

st.header("9. Status Messages")
st.success("Ini pesan sukses!")
st.info("Ini pesan informasi")
st.warning("Ini pesan peringatan")
st.error("Ini pesan error")

st.header("10. Charts")

st.subheader("Line Chart")
chart_data = pd.DataFrame(
    np.random.randn(20, 3),
    columns=["Metrik A", "Metrik B", "Metrik C"]
)
st.line_chart(chart_data)

st.subheader("Bar Chart")
bar_data = pd.DataFrame(
    {"Apel": [10, 25, 18, 30], "Mangga": [15, 12, 22, 8]},
    index=["Jan", "Feb", "Mar", "Apr"]
)
st.bar_chart(bar_data)

st.header("11. Dataframe & Tabel")
data = {
    "Nama":  ["Alice", "Bob", "Charlie", "Diana"],
    "Skor":  [88, 72, 95, 80],
    "Level": ["A", "B", "A+", "A"],
}
df = pd.DataFrame(data)
st.dataframe(df)

Overwriting streamlit_app_basic.py


In [14]:
proc = run_streamlit("streamlit_app_basic.py")

t=2026-05-10T12:34:51+0700 lvl=warn msg="failed to start tunnel" pg=/api/tunnels id=edaf4053f746ca7a err="failed to start tunnel: The endpoint 'https://tributary-spider-disarm.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"


Gagal koneksi ngrok. Streamlit berjalan di http://localhost:8501


In [23]:
# Hentikan app sebelumnya sebelum menjalankan app baru
import os, signal
try:
    proc.terminate()
    print("App dihentikan.")
except:
    pass

App dihentikan.


# Chatbot w gemini


In [20]:
%%writefile streamlit_chat_app.py
import streamlit as st
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()
default_api_key = os.getenv('GOOGLE_API_KEY', '')

st.title("Gemini Chatbot")
st.caption("Chatbot sederhana menggunakan Google Gemini Flash")

with st.sidebar:
    st.subheader("Pengaturan")
    google_api_key = st.text_input("Google AI API Key", value=default_api_key, type="password")
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    st.stop()

if ("genai_client" not in st.session_state) or (getattr(st.session_state, "_last_key", None) != google_api_key):
    try:
        st.session_state.genai_client = genai.Client(api_key=google_api_key)
        st.session_state._last_key = google_api_key
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

if "chat" not in st.session_state:
    st.session_state.chat = st.session_state.genai_client.chats.create(model="gemini-2.5-flash")

if "messages" not in st.session_state:
    st.session_state.messages = []

if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

prompt = st.chat_input("Ketik pesanmu di sini...")

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    try:
        response = st.session_state.chat.send_message(prompt)
        if hasattr(response, "text"):
            answer = response.text
        else:
            answer = str(response)
    except Exception as e:
        answer = f"Terjadi error: {e}"

    with st.chat_message("assistant"):
        st.markdown(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})

Overwriting streamlit_chat_app.py


In [21]:
proc = run_streamlit("streamlit_chat_app.py")

t=2026-05-10T12:36:37+0700 lvl=warn msg="failed to start tunnel" pg=/api/tunnels id=25b36c13a8a0a037 err="failed to start tunnel: The endpoint 'https://tributary-spider-disarm.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"


Gagal koneksi ngrok. Streamlit berjalan di http://localhost:8501


In [19]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.
